# PostgreSQL CDC to Pandas with pydbzengine

This notebook captures Debezium tutorial PostgreSQL change events with `pydbzengine`, stores them in a pandas DataFrame, and shows how many changes were captured per primary key.

## 1) Prerequisites

Start the Debezium tutorial PostgreSQL databse (from `debezium-examples/tutorial`).

Install Python dependencies in your notebook environment:

```bash
pip install pandas pydbzengine psycopg2-binary
```

If you have not run the PostgreSQL tutorial container yet, use the Debezium tutorial defaults.

In [ ]:
import json
import atexit
import os
import threading
import time
from typing import List

import pandas as pd
import ipywidgets as widgets
import psycopg2
from pydbzengine import BasePythonChangeHandler, ChangeEvent, DebeziumJsonEngine
from IPython.display import display


In [ ]:
# Capture target (Debezium tutorial table by default)
TABLE_INCLUDE_LIST = os.getenv('TABLE_INCLUDE_LIST', 'inventory.customers')

# PostgreSQL connection settings (Debezium tutorial defaults)
PG_HOST = os.getenv('PG_HOST', 'localhost')
PG_PORT = os.getenv('PG_PORT', '5432')
PG_USER = os.getenv('PG_USER', 'postgres')
PG_PASSWORD = os.getenv('PG_PASSWORD', 'postgres')
PG_DBNAME = os.getenv('PG_DBNAME', 'postgres')
PG_SQL_USER = os.getenv('PG_SQL_USER', 'postgres')
PG_SQL_PASSWORD = os.getenv('PG_SQL_PASSWORD', 'postgres')
PG_SQL_DBNAME = os.getenv('PG_SQL_DBNAME', 'postgres')

# Debezium connector settings
TOPIC_PREFIX = os.getenv('TOPIC_PREFIX', 'dbserver1')
SCHEMA_INCLUDE_LIST = os.getenv('SCHEMA_INCLUDE_LIST', 'inventory')
SLOT_NAME = os.getenv('SLOT_NAME', 'pydbzengine_slot')
PLUGIN_NAME = os.getenv('PLUGIN_NAME', 'pgoutput')
SNAPSHOT_MODE = os.getenv('SNAPSHOT_MODE', 'initial')  # initial | initial_only | when_needed
PUBLICATION_AUTOCREATE_MODE = os.getenv('PUBLICATION_AUTOCREATE_MODE', 'filtered')

# Local state files used by embedded engine
OFFSET_FILE = os.getenv('OFFSET_FILE', '/tmp/pydbzengine-postgres-offsets.dat')

# Demo behavior
CAPTURE_SECONDS = int(os.getenv('CAPTURE_SECONDS', '20'))
RUN_SAMPLE_SQL = os.getenv('RUN_SAMPLE_SQL', 'true').lower() == 'true'
SAMPLE_PK = int(os.getenv('SAMPLE_PK', '1001'))


In [ ]:
dbz_props = {
    'name': 'pydbzengine-postgres-to-pandas',
    'connector.class': 'io.debezium.connector.postgresql.PostgresConnector',
    'database.hostname': PG_HOST,
    'database.port': PG_PORT,
    'database.user': PG_USER,
    'database.password': PG_PASSWORD,
    'database.dbname': PG_DBNAME,
    'topic.prefix': TOPIC_PREFIX,
    'schema.include.list': SCHEMA_INCLUDE_LIST,
    'table.include.list': TABLE_INCLUDE_LIST,
    'slot.name': SLOT_NAME,
    'plugin.name': PLUGIN_NAME,
    'publication.autocreate.mode': PUBLICATION_AUTOCREATE_MODE,
    'include.schema.changes': 'false',
    'snapshot.mode': SNAPSHOT_MODE,
    'offset.storage': 'org.apache.kafka.connect.storage.FileOffsetBackingStore',
    'offset.storage.file.filename': OFFSET_FILE,
    'offset.flush.interval.ms': '1000'
}

dbz_props


In [ ]:
raw_records = []
raw_records_lock = threading.Lock()

def _as_python_str(value):
    if value is None:
        return None
    return str(value)


def records_dataframe():
    with raw_records_lock:
        rows = list(raw_records)

    df = pd.DataFrame(rows)
    if not df.empty and 'ts_ms' in df.columns:
        df['captured_at'] = pd.to_datetime(df['ts_ms'], unit = 'ms', errors = 'coerce')
    return df


class DataFrameCollector(BasePythonChangeHandler):
    def handleJsonBatch(self, records: List[ChangeEvent]):
        rows = []
        for record in records:
            try:
                key_raw = _as_python_str(record.key())
                key_json = json.loads(key_raw if key_raw else '{}')
                value_raw = _as_python_str(record.value())

                if not value_raw:
                    rows.append({
                        'destination': _as_python_str(record.destination()),
                        'pk': json.dumps(key_json, sort_keys = True),
                        'op': 't',
                        'ts_ms': None,
                        'before': None,
                        'after': None,
                    })
                    continue

                value_json = json.loads(value_raw)
                payload = value_json.get('payload', {})

                rows.append({
                    'destination': _as_python_str(record.destination()),
                    'pk': json.dumps(key_json, sort_keys = True),
                    'op': payload.get('op'),
                    'ts_ms': payload.get('ts_ms'),
                    'before': payload.get('before'),
                    'after': payload.get('after'),
                })
            except Exception as exc:
                # Keep engine alive even if one record is malformed.
                print(f'Skipping malformed record: {exc}')

        if rows:
            with raw_records_lock:
                raw_records.extend(rows)


In [ ]:
engine = None
engine_thread = None

def run_sample_changes(pk: int = SAMPLE_PK):
    sql_statements = [
        'SET search_path TO inventory',
        f'UPDATE customers SET first_name = first_name || \'-x\' WHERE id = {pk}'
    ]
    conn = psycopg2.connect(
        host = PG_HOST,
        port = int(PG_PORT),
        user = PG_SQL_USER,
        password = PG_SQL_PASSWORD,
        dbname = PG_SQL_DBNAME,
    )
    conn.autocommit = True
    try:
        with conn.cursor() as cursor:
            for sql in sql_statements:
                cursor.execute(sql)
    finally:
        conn.close()

def start_engine():
    global engine, engine_thread
    if engine_thread is not None and engine_thread.is_alive():
        print('Engine is already running in the background.')
        return

    # Engine run method is blocking so it must be invoked in its own thread
    engine = DebeziumJsonEngine(properties = dbz_props, handler = DataFrameCollector())
    engine_thread = threading.Thread(target = engine.run, daemon = True)
    engine_thread.start()
    print('Engine started in background.')

def stop_engine():
    global engine, engine_thread
    if engine is None:
        print('Engine is not running.')
        return
    engine.close()
    if engine_thread is not None:
        engine_thread.join(timeout = 10)
    print('Engine stopped.')

# Ensure engine is closed when kernel shuts down.
atexit.register(stop_engine)


In [ ]:
# Use UI buttons to control engine
status = widgets.HTML(value = '<b>Engine status:</b> stopped')

start_btn = widgets.Button(description = 'Start Engine', button_style = 'success')
stop_btn = widgets.Button(description = 'Stop Engine', button_style = 'danger')
sample_btn = widgets.Button(description = 'Run Sample SQL', button_style = 'info')

def on_start_clicked(_):
    start_engine()
    status.value = '<b>Engine status:</b> running'

def on_stop_clicked(_):
    stop_engine()
    status.value = '<b>Engine status:</b> stopped'

def on_sample_clicked(_):
    run_sample_changes()

start_btn.on_click(on_start_clicked)
stop_btn.on_click(on_stop_clicked)
sample_btn.on_click(on_sample_clicked)

display(widgets.HBox([start_btn, stop_btn, sample_btn]))
display(status)


In [ ]:
# Records coming from initial snapshot or from streaming
records_df = records_dataframe()
print(f'Captured {len(records_df)} records so far')
records_df.tail(20)


## 2) Controls

Use the button cell to start and stop the engine, and to generate sample SQL changes.

You can also call these manually in new code cells:

```python
start_engine()
run_sample_changes()
stop_engine()
```


In [ ]:
run_sample_changes(1001)
run_sample_changes(1002)

In [ ]:
# Count actual data changes per primary key.
# Debezium operation codes: c = create, u = update, d = delete, r = read(snapshot), t = tombstone
INCLUDE_SNAPSHOT_READS = False

ops = ['c', 'u', 'd']
if INCLUDE_SNAPSHOT_READS:
    ops.append('r')

working_df = records_dataframe()

if working_df.empty or 'op' not in working_df.columns or 'pk' not in working_df.columns:
    change_counts = pd.DataFrame(columns = ['pk', 'change_count'])
else:
    normalized_ops = working_df['op'].fillna('').astype(str)
    change_counts = (
        working_df[normalized_ops.isin(ops)]
        .groupby('pk', as_index = False)
        .size()
        .rename(columns = {'size': 'change_count'})
        .sort_values('change_count', ascending = False)
    )

change_counts
